# 미관측 도로 기상값 추정 검토

기온·습도·풍속의 지역 제외 결과입니다. 노면온도·결빙 모델은 변경하지 않습니다. 기본 실행은 집계 확인과 저장 모형의 10분 추론이며, 마지막 셀의 `RUN_REFIT=True`는 대표 3모델을 메모리에서 재학습합니다. 원 비공개 프로젝트의 별도 연구 입력이 필요합니다. [설계·출처·한계](../docs/research/ROAD_WEATHER_TRANSFER_20261031.md)를 먼저 확인합니다.

In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import joblib
from threadpoolctl import threadpool_limits
ROOT = Path.cwd() if (Path.cwd()/'src').is_dir() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from src.data_source import sha256_file
from src.road_weather_transfer import features, baselines, constrain, balanced_weights
from src.offline_runtime import deny_network
study = ROOT/'outputs/private/road_weather_transfer_20261031_v1'
summary = json.loads((study/'summary.json').read_text('utf-8'))
audit = json.loads((study/'independent_verification.json').read_text('utf-8'))
assert audit['status'] == 'passed' and sha256_file(study/'summary.json') == audit['summary_sha256']
settings = json.loads((study/'plan.json').read_text('utf-8'))['settings']
metrics = pd.read_csv(ROOT/'outputs/private/road_weather_review_20261031_v1/observation_estimation_metrics.csv')
metrics.loc[metrics.scope.eq('all'), ['target','model','n','mae','rmse','bias','correlation','r2']]

## 동일 시각의 관측과 추론

학습에서 제외된 호남 백양 지역을 예로 사용합니다. 입력 함수는 허용된 기상 열만 선택합니다. 기본 재추론 72행은 저장된 전체 지역 평가의 앞부분이며 새 평가가 아닙니다.

In [ ]:
receipt = json.loads((ROOT/settings['input_output']/'receipt.json').read_text('utf-8'))
data_path = ROOT/settings['input_output']/'development.parquet'
assert sha256_file(data_path) == receipt['data_sha256']
f = pd.read_parquet(data_path).sort_values(['station','time']).reset_index(drop=True)
x = features(f)
examples = []
for target in settings['targets']:
    a, b = baselines(f, target)
    good = f[target].notna() & np.isfinite(a) & np.isfinite(b)
    q = f.loc[f.group.eq('honam_baekyang') & good].head(72)
    model = joblib.load(study/target/'residual_hgb_honam_baekyang.joblib')
    with deny_network() as attempts, threadpool_limits(limits=2):
        p = constrain(model.predict(x.loc[q.index]) + b[q.index], target)
    assert not attempts
    saved = pd.read_parquet(study/target/'predictions.parquet')
    np.testing.assert_array_equal(p, saved.loc[saved.group.eq('honam_baekyang'),'residual_hgb'].head(72))
    example = q[['station','time']].copy()
    example['target'] = target
    example['observed'] = q[target]
    example['prediction'] = p
    examples.append(example)
pd.concat(examples).groupby('target').head(6)

## 선택 실행: 대표 모형 재학습

전체 두 겨울에서 호남 백양을 제외하고, 지점·겨울별 가중치를 같게 맞춰 세 잔차 모형을 다시 적합합니다. 원 파일에 쓰지 않으며 기존 추론과 정확히 대조합니다. 전체 24회 적합과 별도 검산 코드는 `scripts/evaluate_road_weather_transfer.py`, `scripts/audit_road_weather_transfer.py`에 있습니다.

In [ ]:
RUN_REFIT = False
if RUN_REFIT:
    from sklearn.ensemble import HistGradientBoostingRegressor
    for target in settings['targets']:
        a, b = baselines(f, target)
        good = f[target].notna() & np.isfinite(a) & np.isfinite(b)
        train = f.loc[f.group.ne('honam_baekyang') & good]
        q = f.loc[f.group.eq('honam_baekyang') & good]
        model = HistGradientBoostingRegressor(**settings['model'])
        with deny_network() as attempts, threadpool_limits(limits=2):
            model.fit(x.loc[train.index], train[target].to_numpy()-b[train.index], sample_weight=balanced_weights(train))
            p = constrain(model.predict(x.loc[q.index])+b[q.index], target)
        assert not attempts
        saved = pd.read_parquet(study/target/'predictions.parquet')
        np.testing.assert_array_equal(p, saved.loc[saved.group.eq('honam_baekyang'),'residual_hgb'])
        print(target, len(q), '재학습 일치')
else:
    print('기본 실행 완료. 전체 독립 검산에서 대표 3모델 재학습을 별도로 확인했습니다.')